# Qwen3 CoT validation with completion tracking
Use a GPU runtime and upload the updated repository first. T4 uses FP16; the token cap is 1,024. Google Drive stores every trace and its stop reason immediately.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
import subprocess
import sys
from google.colab import drive

os.chdir("/content/PSSN-LLM-Reasoning")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
drive.mount("/content/drive")

from src.utils.environment import collect_environment_info
from src.data.gsm8k import load_gsm8k
from src.models.qwen import GenerationSettings, QwenGenerator
from src.baseline.cot import generate_cot_record, BASELINE_PROMPT_TEMPLATE
from src.utils import ExperimentLogger

SEED = 42
DEVELOPMENT_SIZE = 10
NUMBER_OF_PROBLEMS = 3
MAX_NEW_TOKENS = 1024
assert 0 < NUMBER_OF_PROBLEMS <= DEVELOPMENT_SIZE

environment = collect_environment_info()
assert environment["cuda_available"], "Select a Colab GPU runtime."
if environment["gpus"][0]["compute_capability"] == "7.5":
    assert environment["selected_dtype"] == "float16", "Upload the updated src/ files and restart the runtime."
print(json.dumps(environment, indent=2))

gsm8k = load_gsm8k(development_size=DEVELOPMENT_SIZE, seed=SEED)
settings = GenerationSettings(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, seed=SEED)
generator = QwenGenerator.from_pretrained(settings=settings)
print("Loaded precision:", generator.precision)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path("/content/drive/MyDrive/PSSN2/artifacts/cot_baseline") / stamp
logger = ExperimentLogger(
    output_dir / "cot_baseline_results.jsonl",
    experiment_name="qwen3-4b-gsm8k-cot-validation",
    run_metadata={"dataset": "openai/gsm8k", "subset_source": "train", "seed": SEED, "development_size": DEVELOPMENT_SIZE},
)
records = []
for index, example in enumerate(gsm8k["development"].select(range(NUMBER_OF_PROBLEMS)), start=1):
    record = generate_cot_record(example, generator)
    if not record.get("finish_reason"):
        cap = record.get("generation_config", {}).get("max_new_tokens", MAX_NEW_TOKENS)
        record["finish_reason"] = "length" if record["output_tokens"] >= cap else "eos"
    records.append(logger.log(record))
    print(f"{index}/{NUMBER_OF_PROBLEMS}: {record['problem_id']} | {record['output_tokens']} tokens | stop={record.get('finish_reason', 'unknown')}")

configuration = {
    "run_id": logger.run_id,
    "captured_at_utc": logger.timestamp_utc,
    "repository_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip(),
    "environment": environment,
    "packages": {name: version(name) for name in ["torch", "transformers", "datasets", "accelerate"]},
    "model_metadata": records[0]["model_metadata"],
    "model_revision": getattr(generator.model.config, "_commit_hash", None),
    "tokenizer_revision": generator.tokenizer.init_kwargs.get("_commit_hash"),
    "generation": settings.to_dict(),
    "prompt_template": BASELINE_PROMPT_TEMPLATE,
    "development_size": DEVELOPMENT_SIZE,
    "seed": SEED,
    "dataset_fingerprint": gsm8k["development"]._fingerprint,
    "evaluated_example_ids": [r["problem_id"] for r in records],
    "scoring_policy": "Exclude incomplete token-limit stops; report scored accuracy and excluded count",
}
(output_dir / "run_configuration.json").write_text(json.dumps(configuration, indent=2) + "\n", encoding="utf-8")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
(output_dir / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print("Saved run:", output_dir)